# Dummy agent

Agent loop done by hand, no framework. Local model with Ollama (`qwen2:7b`).

Each cell = one step of the loop:
- system prompt with the tool + Thought / Action / Observation format
- without `stop`, the model invents its own Observation
- with `stop=["Observation:"]`, it stops after the Action
- run the tool manually, send the result back as Observation
- final answer

In [13]:
from openai import OpenAI

# Local Ollama server (OpenAI-compatible endpoint)
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")  # key not checked by Ollama

output = client.chat.completions.create(
    model="qwen2:7b",
    messages=[{"role": "user", "content": "The capital of France is"}],
    max_tokens=100,
)
print(output.choices[0].message.content)

Paris.


In [14]:
# System prompt: tool descriptions + Thought/Action/Observation format
SYSTEM_PROMPT = """Answer the following questions as best you can. You have access to the following tools:

get_weather: Get the current weather in a given location

The way you use the tools is by specifying a json blob.
Specifically, this json should have an `action` key (with the name of the tool to use) and an `action_input` key (with the input to the tool going here).

The only values that should be in the "action" field are:
get_weather: Get the current weather in a given location, args: {"location": {"type": "string"}}
example use :

{
  "action": "get_weather",
  "action_input": {"location": "New York"}
}

ALWAYS use the following format:

Question: the input question you must answer
Thought: you should always think about one action to take. Only one action at a time in this format:
Action:

$JSON_BLOB (inside markdown cell)

Observation: the result of the action. This Observation is unique, complete, and the source of truth.
(this Thought/Action/Observation can repeat N times, you should take several steps when needed. The $JSON_BLOB must be formatted as markdown and only use a SINGLE action at a time.)

You must always end your output with the following format:

Thought: I now know the final answer
Final Answer: the final answer to the original input question

Now begin! Reminder to ALWAYS use the exact characters `Final Answer:` when you provide a definitive answer."""

In [15]:
# Conversation sent to the model: rules + tools (system) and the question (user)
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What's the weather in London?"},
]

print(messages)

[{'role': 'system', 'content': 'Answer the following questions as best you can. You have access to the following tools:\n\nget_weather: Get the current weather in a given location\n\nThe way you use the tools is by specifying a json blob.\nSpecifically, this json should have an `action` key (with the name of the tool to use) and an `action_input` key (with the input to the tool going here).\n\nThe only values that should be in the "action" field are:\nget_weather: Get the current weather in a given location, args: {"location": {"type": "string"}}\nexample use :\n\n{\n  "action": "get_weather",\n  "action_input": {"location": "New York"}\n}\n\nALWAYS use the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about one action to take. Only one action at a time in this format:\nAction:\n\n$JSON_BLOB (inside markdown cell)\n\nObservation: the result of the action. This Observation is unique, complete, and the source of truth.\n(this Thought/

In [16]:
# First call without a stop sequence
output = client.chat.completions.create(
    model="qwen2:7b",
    messages=messages,
    max_tokens=200,
)
print(output.choices[0].message.content)

Question: What's the weather in London?
Thought: I need to get the current weather in London.
Action:

```
{
  "action": "get_weather",
  "action_input": {"location": "London"}
}
```

Observation: (The API response would contain the current weather details for London, such as temperature, humidity, wind speed, and sky conditions.)

Thought: I now know the final answer
Final Answer: The current weather in London is [temperature] degrees Celsius with [description of the sky conditions], the humidity is [humidity]%, and the wind speed is [wind speed] km/h.


We can observe that the Final Answer doesn't make any sense as the LLM doesn't wait the result of the observation before continue generating text.

We will put a stop when he start writing "Observation:"

In [17]:
# Stop generation before the model writes its own Observation
output = client.chat.completions.create(
    model="qwen2:7b",
    messages=messages,
    max_tokens=150,
    stop=["Observation:"],
)
print(output.choices[0].message.content)

Thought: I need to use the get_weather tool to find out the current weather in London.

Action:

```
{
  "action": "get_weather",
  "action_input": {"location": "London"}
}
```




We define the tool that the LLM will use in his action

In [18]:
# Dummy tool: a real implementation would call a weather API
def get_weather(location):
    return f"the weather in {location} is sunny with a temperature of 20°C. \n"

get_weather("London")

'the weather in London is sunny with a temperature of 20°C. \n'

In [19]:
# Model output (Thought + Action) followed by the real tool result as Observation
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What's the weather in London?"},
    {"role": "assistant", "content": output.choices[0].message.content + "Observation:\n" + get_weather("London")},
]

output = client.chat.completions.create(
    model="qwen2:7b",
    messages=messages,
    max_tokens=200,
    stop=["Observation:"],
)
print(output.choices[0].message.content)

Thought: I now know the final answer
Final Answer: The weather in London is sunny with a temperature of 20°C.
